In [2]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "fitbit_no_condition" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
dataset_81383123_fitbit_sleep_daily_summary_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_activity_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_plus_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_steps_intraday = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_daily_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_level = 1 ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (4274025)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (135930, 4101796, 4117930, 255919, 444112, 432453, 4024567, 4274025, 376106, 4011630, 4213101, 4038502, 4093991, 40480457, 4041284, 4042141, 43021226, 4041283, 4227253, 4023995, 4102111, 138239, 4293175, 4212577, 4302537, 4021667)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
fitbit_sleep_daily_summary_81383123_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "fitbit_sleep_daily_summary_81383123",
  "fitbit_sleep_daily_summary_81383123_*.csv")
message(str_glue('The data will be written to {fitbit_sleep_daily_summary_81383123_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_81383123_fitbit_sleep_daily_summary_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  fitbit_sleep_daily_summary_81383123_path,
  destination_format = "CSV")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_81383123/fitbit_sleep_daily_summary_81383123_*.csv. Use this path when reading the data into your notebooks in the future.



In [21]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_sleep_daily_summary_81383123_path}` to copy these files
#       to the Jupyter disk.
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
fitbit_sleep_daily_summary_81383123_path <- "gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_81383123/fitbit_sleep_daily_summary_81383123_*.csv"

healthy_sleep <- read_bq_export_from_workspace_bucket(fitbit_sleep_daily_summary_81383123_path)

dim(healthy_sleep)

head(healthy_sleep, 5)


Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_81383123/fitbit_sleep_daily_summary_81383123_000000000000.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_81383123/fitbit_sleep_daily_summary_81383123_000000000001.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_81383123/fitbit_sleep_daily_summary_81383123_000000000002.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_81383123/fitbit_sleep_daily_summary_81383123_000000000003.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_81383123/fitbit_sleep_daily_summary_81383123_000000000004.csv.

Loading gs://fc-secu

[1] 14014835       12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2095035,2020-06-07,true,553,466,0,87,NA,64,289,113,87
2073612,2022-04-01,true,561,466,0,95,NA,75,312,79,95
2075212,2020-10-20,true,549,467,0,82,NA,72,303,92,82
2080601,2022-03-17,true,550,467,0,83,NA,76,283,108,83
2062616,2021-07-05,true,553,467,7,86,NA,93,298,76,86


In [1]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


corrplot 0.95 loaded


Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object

In [23]:
colnames(healthy_sleep)[2] <- 'date'
healthy_sleep_main <- healthy_sleep |>
  filter(is_main_sleep == "true")

healthy_sleep_nap <- healthy_sleep |>
  filter(is_main_sleep == "false")

In [2]:
#colnames(healthy_sleep)[2] <- 'date'
fitbit_sleep_daily_summary_81383123_path <- "gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_81383123/fitbit_sleep_daily_summary_81383123_*.csv"

healthy_sleep_sum <- sleep_pipeline_from_gcs(
    path.out =  '/home/jupyter/workspaces/result/Healthy/',
    export_path = fitbit_sleep_daily_summary_81383123_path,
    cohort = 'Healthy',
    main = "sum")

Found 10 CSV files.
File 1 / 10 : fitbit_sleep_daily_summary_81383123_000000000000.csv 
File 2 / 10 : fitbit_sleep_daily_summary_81383123_000000000001.csv 
File 3 / 10 : fitbit_sleep_daily_summary_81383123_000000000002.csv 
File 4 / 10 : fitbit_sleep_daily_summary_81383123_000000000003.csv 
File 5 / 10 : fitbit_sleep_daily_summary_81383123_000000000004.csv 
File 6 / 10 : fitbit_sleep_daily_summary_81383123_000000000005.csv 
File 7 / 10 : fitbit_sleep_daily_summary_81383123_000000000006.csv 
File 8 / 10 : fitbit_sleep_daily_summary_81383123_000000000007.csv 
File 9 / 10 : fitbit_sleep_daily_summary_81383123_000000000008.csv 
File 10 / 10 : fitbit_sleep_daily_summary_81383123_000000000009.csv 
Deduplicating daily summary...
Computing patient stats...
            metric     N      Mean        SD    Median        Q1        Q3
1    minute_in_bed 22970 434.48416 89.506477 450.41958 400.47816 488.58892
2    minute_asleep 22970 382.11775 78.791169 395.73162 350.41988 430.45509
3     minute_awa

In [20]:
healthy_sleep_sum[1:3,]

person_id,n_obs,minute_in_bed__mean,minute_in_bed__sd,minute_in_bed__cv,minute_asleep__mean,minute_asleep__sd,minute_asleep__cv,minute_awake__mean,minute_awake__sd,⋯,minute_rem__mean,minute_rem__sd,minute_rem__cv,minute_wake__mean,minute_wake__sd,minute_wake__cv,sleep_efficiency__mean,sleep_efficiency__sd,sleep_efficiency__cv,cohort
<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
2095035,1173,370.9787,140.5766,37.89343,323.3615,118.8505,36.75470,47.50810,25.27489,⋯,58.16198,36.48127,62.72358,46.87212,26.22508,55.95028,88.31284,4.614903,5.225631,Healthy
2073612,2614,396.0803,119.9764,30.29092,349.4151,105.9469,30.32120,45.44262,19.76960,⋯,62.66756,47.00146,75.00126,36.98317,27.71800,74.94761,88.43245,3.427282,3.875593,Healthy
2075212,2672,460.0445,144.7754,31.46987,408.2201,127.1823,31.15532,51.22418,26.32944,⋯,68.99701,54.89512,79.56160,42.93750,32.81488,76.42476,89.07389,4.722485,5.301761,Healthy


In [28]:
colnames(healthy_sleep)[2] <- 'date'
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/Healthy/sleep/', df = healthy_sleep_main)

── Wear compliance ──
     n_days       followup_days   
 Min.   :   1.0   Min.   :   1.0  
 1st Qu.:  61.0   1st Qu.: 105.0  
 Median : 187.0   Median : 514.0  
 Mean   : 531.6   Mean   : 952.7  
 3rd Qu.: 770.0   3rd Qu.:1589.0  
 Max.   :3992.0   Max.   :4964.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1000107   Min.   :   1.0   Min.   :2009-12-19   Min.   :2013-09-02  
 1st Qu.:2099866   1st Qu.:  61.0   1st Qu.:2018-04-21   1st Qu.:2023-04-20  
 Median :3769178   Median : 187.0   Median :2021-09-22   Median :2023-09-26  
 Mean   :4541989   Mean   : 531.6   Mean   :2020-08-12   Mean   :2023-03-22  
 3rd Qu.:6864914   3rd Qu.: 770.0   3rd Qu.:2023-06-08   3rd Qu.:2023-10-01  
 Max.   :9999996   Max.   :3992.0   Max.   :2023-10-01   Max.   :2023-10-01  
 followup_days     missing_days   pct_missing       compliance_tier
 Min.   :   1.0   Min.   :   0   Min.   : 0.00   High (>90%):5683  
 1st Qu.: 105.0   1st Qu.:  21   1st Qu.:10.22   80-90%     :3241  
 Median : 514.0   Median :  86   Median :31.69   70-80%     :2237  
 Mean   : 952.7   Mean   : 421   Mean   :38.42   60-70%     :2028  
 3rd Qu.:1589.0   3rd Qu.: 506   3rd Qu.:64.10   50-60%     :1748  
 Max.   :4964.0   Max.   :4747   Max.   :99.88

NULL

In [8]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/Data/',
    cohort = 'Healthy',
    df = healthy_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/Data/',
    cohort = 'Healthy',
    df = healthy_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep 22970 368.   79.8  382.  333   420.  204.  468. 
2 minute_awake  22970  49.4  17.6   50.1  40.3  59.2  18.0  74.5
3 minute_deep   22225  60.0  16.6   60.2  48.6  71.4  33    86.9
4 minute_in_bed 22970 418.   90.5  435.  380.  476.  230.  531. 
5 minute_light  22225 251.   43.6  250.  223.  278.  182.  323. 
6 minute_rem    22225  79.1  21.4   80.0  65.0  94.0  42.3 112. 
7 minute_wake   22225  55.7  11.5   55.3  48.3  62.8  37.7  75.0
# A tibble: 7 × 9
  metric            N  Mean    SD Median     Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl>  <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep 21348 105.  30.3  102.    87.8  118.   65.3  150. 
2 minute_awake  21348  12.2  9.40   9.23   6.63  13.7   3.4   33.7
3 minute_deep   14950  25.5 11.4   25     18.3   32     7.43  45  
4 minute_in_bed 21348 119.  31.8  115.   100

In [3]:
head(healthy_sleep_sum)
patient_stats_sum <- sleep_prep_stats(df = healthy_sleep_sum, path.out =  '/home/jupyter/workspaces/result/Healthy/', cohort = 'Healthy', main = "sum")

person_id,n_days,pct_days_with_nap,pct_days_with_mainsleep,minute_in_bed__mean,minute_in_bed__sd,minute_in_bed__cv,minute_asleep__mean,minute_asleep__sd,minute_asleep__cv,⋯,minute_rem__sd,minute_rem__cv,minute_wake__mean,minute_wake__sd,minute_wake__cv,sleep_efficiency__mean,sleep_efficiency__sd,sleep_efficiency__cv,sleep_efficiency_overall,cohort
<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
2095035,1032,0,0,421.6647,97.68315,23.16607,367.5417,82.89976,22.55520,⋯,31.74211,48.01515,53.27616,21.22514,39.83984,87.52840,3.598325,4.111037,87.16443,Healthy
2073612,2371,0,0,436.6740,69.11314,15.82717,385.2261,62.17578,16.14008,⋯,44.69433,64.68978,40.77351,26.34998,64.62523,88.23917,3.098150,3.511082,88.21823,Healthy
2075212,2480,0,0,495.6609,122.74515,24.76394,439.8242,107.46421,24.43345,⋯,54.19893,72.90808,46.26169,32.42102,70.08179,88.89810,4.023812,4.526320,88.73490,Healthy
2080601,584,0,0,411.8288,181.57922,44.09095,362.7192,157.16641,43.33005,⋯,54.35131,85.27757,43.33904,31.40634,72.46663,88.89175,4.001158,4.501158,88.07524,Healthy
2062616,1790,0,0,492.3983,70.58124,14.33417,436.1514,64.69263,14.83261,⋯,40.11106,63.02839,50.36592,32.83386,65.19062,88.67891,4.830878,5.447607,88.57695,Healthy
2069997,693,0,0,479.6335,82.02971,17.10258,419.8600,74.54267,17.75417,⋯,31.35388,37.86331,58.82107,15.29677,26.00559,87.48371,2.737117,3.128717,87.53768,Healthy


ERROR: [1m[33mError[39m in `summarise()`:[22m
[1m[22m[36mℹ[39m In argument: `across(...)`.
[1mCaused by error in `across()`:[22m
[1m[22m[36mℹ[39m In argument: `all_of(key_metrics)`.
[1mCaused by error in `all_of()`:[22m
[33m![39m Can't subset elements that don't exist.
[31m✖[39m Elements `minute_in_bed`, `minute_asleep`, `minute_awake`, `minute_deep`, `minute_light`, etc. don't exist.


In [27]:
miss_gap_func(df = healthy_sleep_main, path.out =  '/home/jupyter/workspaces/result/Healthy/sleep/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    0.0     7.0    27.0   171.8   127.0  4743.0 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0      22157         96.5        813         3.54
2         7      17095         74.4       5875        25.6 
3        14      13915         60.6       9055        39.4 
4        30      11094         48.3      11876        51.7 
5        60       7997         34.8      14973        65.2 


NULL